Data Visualization

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
import re

print("Memulai pembuatan model Neural Network...")

df = None # Initialize df to None
data_loaded_successful = False

# 0. Muat Data dari CSV
print("\n--- Muat Data dari internship (1).csv ---")
try:
    df = pd.read_csv('internship.csv')
    # Menghilangkan spasi ekstra pada nama kolom jika ada
    df.columns = df.columns.str.strip()
    print("Data berhasil dimuat.")
    print("Nama kolom yang tersedia:", df.columns.tolist())
    data_loaded_successful = True
except FileNotFoundError:
    print("Error: File 'internship (1).csv' tidak ditemukan. Pastikan file sudah diupload.")
except Exception as e:
    print(f"Error saat membaca CSV: {e}")

if not data_loaded_successful:
    print("Tidak dapat melanjutkan karena data gagal dimuat.")
else:
    print("5 Baris Data Sampel:")
    print(df.head())
    print("\n")

    # Initialize data_processing_successful inside the else block
    data_processing_successful = True

    # Fungsi untuk mengonversi 'Stipend' menjadi 'Stipend Label'
    def categorize_stipend(stipend_value):
        if isinstance(stipend_value, str):
            stipend_value = stipend_value.lower().strip()
            if 'unpaid' in stipend_value:
                return 0 # Unpaid
            elif '₹' in stipend_value:
                cleaned_stipend = re.sub(r'[₹,]', '', stipend_value)


                if '–' in cleaned_stipend or '-' in cleaned_stipend:
                    separator = '–' if '–' in cleaned_stipend else '-'
                    parts = cleaned_stipend.split(separator)
                    num_match = re.search(r'\d+', parts[0])
                    if num_match:
                        val = float(num_match.group())
                    else:
                        return -1
                else:
                    cleaned_num_str = re.sub(r'[^0-9.]', '', cleaned_stipend).strip()
                    if cleaned_num_str:
                        val = float(cleaned_num_str)
                    else:
                        return -1
                if val != -1:
                    if val < 10000:
                        return 1 # < ₹10.000
                    elif 10000 <= val <= 20000:
                        return 2 # ₹10.000–20.000
                    else: # val > 20000
                        return 3 # > ₹20.000
        return -1 # Default atau untuk nilai yang tidak terdefinisi

    if 'stipend' in df.columns:
        df['Stipend Label'] = df['stipend'].apply(categorize_stipend)
        initial_rows = len(df)
        df = df[df['Stipend Label'] != -1].reset_index(drop=True)
        if len(df) < initial_rows:
            print(f"Peringatan: {initial_rows - len(df)} baris dihapus karena nilai 'stipend' tidak dapat dikategorikan atau diproses dengan benar.")
        print("Kolom 'Stipend Label' berhasil dibuat dan data yang tidak relevan dihapus.")
        if df.empty:
            print("Error: Tidak ada data yang tersisa setelah pembersihan 'Stipend Label'.")
            data_processing_successful = False
    else:
        print("Error: Kolom 'stipend' tidak ditemukan di dataset. Tidak dapat membuat 'Stipend Label'.")
        data_processing_successful = False

    if data_processing_successful:
        stipend_categories_labels = ['Unpaid', 'Low', 'Medium', 'High'] # 0, 1, 2, 3

        # Tahap 1 Data Understanding
        print("Tahap 1 Data Understanding")
        print("Statistik Deskriptif Dataset:")
        print(df.describe(include='all'))
        print(f"\nJumlah total data: {len(df)}")
        print("\nMissing Values per Kolom:")
        print(df.isnull().sum())

        # Cek duplikasi
        print(f"\nJumlah data duplikat awal: {df.duplicated().sum()}")
        print("\n")

        # Tahap 2 Data Preprocessing
        print("Tahap 2 Data Preprocessing")

        #Menghapus data kosong dan duplikasi
        print("Missing Values sebelum penghapusan:")
        print(df.isnull().sum())

        df_cleaned = df.dropna().reset_index(drop=True)
        print(f"\nJumlah data setelah menghapus baris dengan missing values: {len(df_cleaned)}")

        initial_rows_after_na = len(df_cleaned)
        df_cleaned.drop_duplicates(inplace=True)
        print(f"Jumlah baris duplikat yang dihapus: {initial_rows_after_na - len(df_cleaned)}")
        print(f"Jumlah data setelah menghapus duplikasi: {len(df_cleaned)}")
        print("\n")

        # Konversi kolom 'duration' dari string ke numerik
        if 'duration' in df_cleaned.columns:
            def extract_duration_months(duration_str):
                if isinstance(duration_str, str):
                    match = re.search(r'(\d+)\s*(?:Months?|month?)', duration_str, re.IGNORECASE)
                    if match:
                        return int(match.group(1))
                return np.nan

            original_duration_dtype = df_cleaned['duration'].dtype
            df_cleaned['duration'] = df_cleaned['duration'].apply(extract_duration_months)
            df_cleaned.dropna(subset=['duration'], inplace=True)
            if not df_cleaned.empty:
                df_cleaned['duration'] = df_cleaned['duration'].astype(int)
                print(f"Kolom 'duration' berhasil dikonversi dari {original_duration_dtype} ke int.")
            else:
                print("Error: Tidak ada data yang tersisa setelah pembersihan 'duration'.")
                data_processing_successful = False
        else:
            print("Peringatan: Kolom 'duration' tidak ditemukan, tidak dapat mengonversi.")
        print("\n")

    if data_processing_successful:
        categorical_features = ['internship_title', 'company_name', 'location', 'start_date']
        numerical_features = ['duration']
        target_feature = 'Stipend Label'

        all_features_exist = True
        for col in categorical_features + numerical_features:
            if col not in df_cleaned.columns:
                print(f"Error: Kolom '{col}' tidak ditemukan di DataFrame setelah pembersihan. Harap periksa nama kolom atau data.")
                all_features_exist = False
                break

        if not all_features_exist:
            print("Tidak dapat melanjutkan karena kolom fitur yang dibutuhkan tidak lengkap.")
            data_processing_successful = False

        if data_processing_successful:
            X = df_cleaned[categorical_features + numerical_features]
            y = df_cleaned[target_feature]

            # Transformasi: One-Hot Encoding untuk fitur kategorikal dan Normalisasi: MinMaxScaler untuk fitur numerikal
            preprocessor = ColumnTransformer(
                transformers=[
                    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
                    ('num', MinMaxScaler(), numerical_features)
                ])

            X_processed = preprocessor.fit_transform(X)

            y_processed = to_categorical(y, num_classes=len(stipend_categories_labels))

            print(f"Shape data fitur (X) setelah preprocessing: {X_processed.shape}")
            print(f"Shape data target (y) setelah one-hot encoding: {y_processed.shape}")
            print("\n")

Memulai pembuatan model Neural Network...

--- Muat Data dari internship (1).csv ---
Data berhasil dimuat.
Nama kolom yang tersedia: ['internship_title', 'company_name', 'location', 'start_date', 'duration', 'stipend']
5 Baris Data Sampel:
            internship_title                             company_name  \
0           Java Development                              SunbaseData   
1     Accounting and Finance                          DAKSM & Co. LLP   
2  Sales & Digital Marketing  Bharat Natural Elements Private Limited   
3    Social Entrepreneurship                       Hamari Pahchan NGO   
4  Videography & Photography                        Esquare Lifestyle   

         location   start_date  duration                stipend  
0  Work From Home  Immediately  6 Months        ₹ 30,000 /month  
1           Noida  Immediately  6 Months  ₹ 5,000-10,000 /month  
2       Bangalore  Immediately  6 Months         ₹ 5,000 /month  
3  Work From Home  Immediately  6 Months                 